# 05 - Đánh Giá & Báo Cáo Tổng Hợp

**Mục tiêu:**
- Tổng hợp kết quả từ tất cả các bước
- Rút actionable insights
- So sánh ưu nhược điểm
- Đề xuất hướng phát triển

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

from src.data.loader import load_params
params = load_params()

## 5.1 Tổng hợp kết quả

In [ ]:
# Load kết quả đã lưu
try:
    cls_results = pd.read_csv('outputs/tables/classification_results.csv', index_col=0)
    print("=== CLASSIFICATION RESULTS ===")
    print(cls_results.round(4))
except:
    print("Chưa có kết quả classification")

In [ ]:
try:
    cv_results = pd.read_csv('outputs/tables/cross_validation_results.csv', index_col=0)
    print("\n=== CROSS-VALIDATION RESULTS ===")
    print(cv_results.round(4))
except:
    print("Chưa có kết quả cross-validation")

In [ ]:
try:
    ss_results = pd.read_csv('outputs/tables/semi_supervised_learning_curve.csv')
    print("\n=== SEMI-SUPERVISED RESULTS ===")
    print(ss_results.round(4))
except:
    print("Chưa có kết quả semi-supervised")

In [ ]:
try:
    rules = pd.read_csv('outputs/tables/association_rules_heart.csv', index_col=0)
    print("\n=== TOP ASSOCIATION RULES (Heart Disease) ===")
    print(rules.head(10))
except:
    print("Chưa có kết quả association rules")

## 5.2 Visualization tổng hợp

In [ ]:
# Dashboard tổng hợp
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Classification comparison
if 'cls_results' in dir():
    cls_results[['F1-Score', 'ROC-AUC', 'PR-AUC']].plot(kind='barh', ax=axes[0,0], colormap='Set2')
    axes[0,0].set_title('Model Comparison', fontweight='bold')
    axes[0,0].set_xlim(0, 1)

# 2. FN analysis
if 'cls_results' in dir():
    cls_results['FN Count'].plot(kind='bar', ax=axes[0,1], color='#e74c3c')
    axes[0,1].set_title('False Negatives by Model', fontweight='bold')
    axes[0,1].tick_params(axis='x', rotation=45)

# 3. Semi-supervised learning curve
if 'ss_results' in dir():
    axes[1,0].plot(ss_results['label_ratio']*100, ss_results['supervised_f1'], 'bo-', label='Supervised')
    axes[1,0].plot(ss_results['label_ratio']*100, ss_results['self_training_f1'], 'rs-', label='Self-Training')
    axes[1,0].plot(ss_results['label_ratio']*100, ss_results['label_spreading_f1'], 'g^-', label='Label Spreading')
    axes[1,0].set_xlabel('% Labeled Data')
    axes[1,0].set_ylabel('F1 Score')
    axes[1,0].set_title('Semi-Supervised Learning Curve', fontweight='bold')
    axes[1,0].legend()

# 4. Association rules (top rules by lift)
if 'rules' in dir() and len(rules) > 0:
    top_rules = rules.head(8)
    axes[1,1].barh(range(len(top_rules)), top_rules['lift'], color=sns.color_palette('viridis', len(top_rules)))
    axes[1,1].set_yticks(range(len(top_rules)))
    axes[1,1].set_yticklabels([a[:30] for a in top_rules['antecedents']], fontsize=8)
    axes[1,1].set_xlabel('Lift')
    axes[1,1].set_title('Top Association Rules (Lift)', fontweight='bold')

plt.suptitle('DASHBOARD - Heart Disease Data Mining Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/dashboard_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Phân tích Lỗi Chi tiết (Error Pattern Analysis)

> ⚠️ **Tiêu chí G yêu cầu phân tích lỗi và ≥ 5 actionable insights**

In [ ]:
# Chạy lại model tốt nhất để phân tích lỗi
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_clean = pd.read_csv(params['paths']['processed_data'])
from src.features.builder import select_features_for_modeling
X, y = select_features_for_modeling(df_clean)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=params['split']['test_size'],
    stratify=y, random_state=params['seed']
)

from src.models.supervised import train_and_evaluate
results, results_df = train_and_evaluate(
    X_train, X_test, y_train, y_test,
    use_smote=params['classification']['use_smote'],
    random_state=params['seed']
)

best_name = results_df.index[0]
print(f"Best model: {best_name}")

In [ ]:
# Phân tích lỗi theo nhóm tuổi, giới tính, loại đau ngực
from src.evaluation.metrics import analyze_error_patterns

error_analysis = analyze_error_patterns(
    X_test, y_test, results[best_name]['y_pred'],
    feature_names=X.columns.tolist()
)

In [ ]:
# Confusion Matrix chi tiết
from src.visualization.plots import plot_confusion_matrix
fig = plot_confusion_matrix(
    error_analysis['confusion_matrix'],
    title=f'Error Analysis - {best_name}'
)
plt.show()

## 5.4 Actionable Insights (≥ 7 insights)

In [ ]:
from src.evaluation.metrics import generate_actionable_insights

insights = generate_actionable_insights(results, df_clean)
print("=" * 60)
print("ACTIONABLE INSIGHTS")
print("=" * 60)
for insight in insights:
    print(insight)
    print()

## 5.5 So sánh ưu/nhược từng phương án

| Mô hình | Vai trò | Ưu điểm | Nhược điểm |
|---|---|---|---|
| Dummy Classifier | Baseline | Tham chiếu tối thiểu | Không học gì |
| Logistic Regression | Baseline | Nhanh, interpretable | Giả định tuyến tính |
| SVM (linear) | Cải tiến | Margin tối ưu | Chậm, khó interpret |
| SVM (RBF) | Cải tiến | Non-linear | Nhạy hyperparams |
| Random Forest | Cải tiến | Robust, feature importance | Nhiều hyperparams |
| XGBoost | Cải tiến | State-of-art, fast | Cần tuning cẩn thận |

## 5.6 Thách thức gặp phải

1. Missing values cao (ca: 66%, thal: 53%) → ảnh hưởng chất lượng
2. Dataset gộp từ nhiều nguồn → phân phối khác nhau
3. Chọn ngưỡng rời rạc hoá cho Apriori cần kiến thức y tế
4. Cân bằng giữa Precision và Recall trong bối cảnh y tế
5. FN (bỏ sót bệnh) nguy hiểm hơn FP trong y tế

## 5.7 Tổng kết & Hướng phát triển

### Kết quả đạt được:
- Pipeline Data Mining hoàn chỉnh 6 bước
- Apriori tìm được luật kết hợp có ý nghĩa y học
- Phân cụm xác định nhóm nguy cơ rõ ràng (Silhouette + DBI)
- Classification đạt PR-AUC > 0.85, vượt trội baseline
- Semi-supervised hiệu quả với 20%+ nhãn
- Phân tích lỗi chi tiết theo nhóm tuổi/giới/triệu chứng

### Hướng phát triển:
1. Thu thập thêm dữ liệu, đặc biệt giảm missing values
2. Thử Deep Learning (Neural Network)
3. Triển khai web app (Streamlit) cho demo
4. Kết hợp thêm dữ liệu y tế (ECG signals, imaging)
5. Tối ưu threshold cho từng bối cảnh lâm sàng cụ thể

In [ ]:
# Tạo báo cáo tổng hợp
from src.evaluation.report import create_summary_report

report = create_summary_report(
    eda_stats={'n_rows': 920, 'n_cols': 16, 'total_missing': 1759},
    association_rules_count=len(rules) if 'rules' in dir() else 0,
    clustering_results={'best_k': 3, 'silhouette': 0.15},
    classification_results=cls_results if 'cls_results' in dir() else None,
    semi_supervised_results=ss_results if 'ss_results' in dir() else None,
)
print(report)